# Census-Backed Realistic Profile Distributions

By default, `govsynth` generates profiles using the `edge_saturated` strategy — profiles are
deliberately placed at policy thresholds to stress-test model reasoning at decision boundaries.
That is the right approach for evaluating **rationale quality**.

But it produces a skewed population. Every household sits within a few percent of a program
limit. Real-world SNAP applicants span a much wider income distribution, include households
that are clearly eligible or clearly ineligible, and reflect state-level demographics.

The `realistic` profile strategy addresses this by drawing from Census ACS (American Community
Survey) distributions: income follows a lognormal fit to ACS B19001 income brackets;
household size, citizenship, labor force participation, housing tenure, and age all come from
state-specific ACS tables. The result is a benchmark population that mirrors the actual
applicant pool — which matters for measuring coverage and false-negative rates across the
full income spectrum.

**What this notebook covers:**
1. Load and inspect a `CensusDistribution` directly
2. Compare a single profile from `uniform` vs `realistic`
3. Compare distributions across 200 profiles from each strategy
4. Show how different states produce different profiles
5. Use the `realistic` strategy via the Pipeline
6. How to refresh census data for a new state

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import math
import warnings
from collections import Counter

from govsynth.sources.us.census import CensusDataSource, CensusDistribution
from govsynth.profiles.us_household import USHouseholdProfile

print('govsynth imported successfully')

## 1. Loading a Census Distribution

`CensusDataSource` deserializes the ACS JSON bundle at `data/census/<state>.json` into a
typed `CensusDistribution` dataclass. Virginia is pre-bundled with the package — no network
call needed. Fields map directly to the ACS tables that were fetched; the metadata records
which vintage and survey were used.

In [ ]:
dist = CensusDataSource('VA').load()

print(f'State:                      {dist.state}')
print()
print('--- Income ---')
print(f'  Monthly lognormal mu:     {dist.income_mu}  (ln of monthly dollars)')
print(f'  Monthly lognormal sigma:  {dist.income_sigma}')
implied_median = math.exp(dist.income_mu)
print(f'  Implied median income:    ${implied_median:,.0f}/month  (${implied_median * 12:,.0f}/year)')
print(f'  FPL buckets (spot check): {dist.fpl_buckets[:2]} ...')
print()
print('--- Household ---')
for i, w in enumerate(dist.household_size_weights, start=1):
    bar = '█' * round(w * 40)
    print(f'  Size {i}: {w:.1%}  {bar}')
print()
print('--- Demographics ---')
print(f'  % with children:          {dist.pct_with_children:.1%}')
print(f'  % elderly or disabled:    {dist.pct_elderly_or_disabled:.1%}')
print(f'  % citizen:                {dist.pct_citizen:.1%}')
print(f'  % noncitizen eligible:    {dist.pct_noncitizen_eligible:.1%}')
print(f'  Age (mean ± sd):          {dist.age_mu:.0f} ± {dist.age_sigma:.0f} years')
print()
print('--- Income Sources ---')
print(f'  Labor force participation: {dist.labor_force_participation_rate:.1%}')
print(f'  % receiving Social Security: {dist.pct_social_security:.1%}')
print(f'  % receiving SSI:           {dist.pct_ssi:.1%}')
print(f'  % public assistance:       {dist.pct_public_assistance:.1%}')
print()
print('--- Program Participation ---')
print(f'  SNAP receipt rate:        {dist.snap_receipt_rate:.1%}')
print(f'  Medicaid coverage rate:   {dist.medicaid_coverage_rate:.1%}')
print()
print('--- Housing ---')
print(f'  Median gross rent:        ${dist.median_gross_rent_monthly:,.0f}/month')
print(f'  % renter-occupied:        {dist.pct_renter:.1%}')

## 2. Uniform vs. Realistic — Single Profile

With the same seed, the two strategies produce very different households. `uniform` uses
coarse national approximations; `realistic` draws from the Virginia ACS distributions above.
The key differences to look for: income level, household size weights, and citizenship status.

In [ ]:
uniform   = USHouseholdProfile.random(state='VA', seed=42, strategy='uniform')
realistic = USHouseholdProfile.random(state='VA', seed=42, strategy='realistic')

fields = [
    ('household_size',        'HH Size'),
    ('monthly_gross_income',  'Gross Income/mo'),
    ('has_elderly_or_disabled','Has Elderly/Disabled'),
    ('has_dependent_children','Has Children'),
    ('citizenship_status',    'Citizenship'),
    ('age_of_head',           'Age of Head'),
    ('shelter_costs',         'Shelter Costs/mo'),
    ('liquid_assets',         'Liquid Assets'),
]

print(f'{"Field":<26} {"Uniform":>20} {"Realistic":>20}')
print('-' * 68)
for attr, label in fields:
    u_val = getattr(uniform, attr)
    r_val = getattr(realistic, attr)
    if isinstance(u_val, float):
        u_str = f'${u_val:,.2f}'
        r_str = f'${r_val:,.2f}'
    elif hasattr(u_val, 'value'):
        u_str = u_val.value
        r_str = r_val.value
    else:
        u_str = str(u_val)
        r_str = str(r_val)
    print(f'{label:<26} {u_str:>20} {r_str:>20}')

## 3. Distribution Comparison — 200 Profiles

A single profile doesn't tell the full story. Generating 200 profiles from each strategy
lets us compare the aggregate distributions: income spread, household size mix, and
citizenship composition. The realistic strategy should produce a higher median income
(Virginia skews affluent vs. national) and a citizenship mix matching ACS proportions.

In [ ]:
N = 200

profiles_uniform   = [USHouseholdProfile.random(state='VA', seed=i, strategy='uniform')   for i in range(N)]
profiles_realistic = [USHouseholdProfile.random(state='VA', seed=i, strategy='realistic') for i in range(N)]

print(f'Generated {N} profiles per strategy.')

In [ ]:
def ascii_histogram(values, n_bins=8, width=40, fmt='.0f'):
    """Print an ASCII bar chart of a numeric distribution."""
    lo, hi = min(values), max(values)
    if lo == hi:
        print(f'  All values equal {lo:{fmt}}')
        return
    bin_size = (hi - lo) / n_bins
    counts = [0] * n_bins
    for v in values:
        idx = min(int((v - lo) / bin_size), n_bins - 1)
        counts[idx] += 1
    max_count = max(counts)
    for i, c in enumerate(counts):
        lo_b = lo + i * bin_size
        hi_b = lo_b + bin_size
        bar = '█' * round(c / max_count * width)
        print(f'  ${lo_b:{fmt}}-${hi_b:{fmt}}: {bar:<{width}} {c}')

incomes_u = [p.monthly_gross_income for p in profiles_uniform]
incomes_r = [p.monthly_gross_income for p in profiles_realistic]

sorted_u = sorted(incomes_u)
sorted_r = sorted(incomes_r)
median_u = sorted_u[len(sorted_u) // 2]
median_r = sorted_r[len(sorted_r) // 2]

print('=== INCOME DISTRIBUTION — UNIFORM (national approximation) ===')
print(f'  Median: ${median_u:,.0f}/mo   Min: ${min(incomes_u):,.0f}   Max: ${max(incomes_u):,.0f}')
ascii_histogram(incomes_u, n_bins=8)
print()
print('=== INCOME DISTRIBUTION — REALISTIC (Virginia ACS) ===')
print(f'  Median: ${median_r:,.0f}/mo   Min: ${min(incomes_r):,.0f}   Max: ${max(incomes_r):,.0f}')
ascii_histogram(incomes_r, n_bins=8)

In [ ]:
# Household size distribution
sizes_u = Counter(p.household_size for p in profiles_uniform)
sizes_r = Counter(p.household_size for p in profiles_realistic)

print(f'  {"HH Size":<10} {"Uniform":>10} {"Realistic":>12} {"ACS (VA)":>12}')
print('-' * 48)
for size in range(1, 7):
    u_pct = sizes_u.get(size, 0) / N
    r_pct = sizes_r.get(size, 0) / N
    acs_pct = dist.household_size_weights[size - 1]
    print(f'  {size:<10} {u_pct:>9.1%} {r_pct:>11.1%} {acs_pct:>11.1%}')
print()
print('Realistic weights track ACS; uniform weights are coarse national approximations.')

In [ ]:
# Citizenship mix
cit_u = Counter(p.citizenship_status.value for p in profiles_uniform)
cit_r = Counter(p.citizenship_status.value for p in profiles_realistic)
all_statuses = sorted(set(cit_u) | set(cit_r))

print(f'  {"Status":<30} {"Uniform":>10} {"Realistic":>12}')
print('-' * 55)
for status in all_statuses:
    u_pct = cit_u.get(status, 0) / N
    r_pct = cit_r.get(status, 0) / N
    print(f'  {status:<30} {u_pct:>9.1%} {r_pct:>11.1%}')
print()
print(f'ACS VA: {dist.pct_citizen:.1%} citizen, {dist.pct_noncitizen_eligible:.1%} noncitizen-eligible')
print('Uniform strategy does not vary citizenship — realistic strategy mirrors ACS.')

## 4. State Comparison — VA, CA, TX

State-level ACS data reveals meaningful demographic differences that affect how a
realistic benchmark population looks. Virginia has high median incomes; Texas has a larger
renter population and higher SNAP participation; California has the highest median rent.

Only VA is pre-bundled. CA and TX will fall back gracefully to national weights if their
census files are not present — the comparison still runs, but CA and TX numbers will
reflect national approximations rather than state-specific ACS.

In [ ]:
states = ['VA', 'CA', 'TX']
state_dists = {}

for state in states:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        d = CensusDataSource(state).load()
        if caught:
            print(f'  {state}: WARNING — {caught[0].message} (using national fallback)')
        else:
            print(f'  {state}: loaded state-specific ACS data')
    state_dists[state] = d

print()

In [ ]:
# Generate 100 realistic profiles per state and compare key statistics
state_profiles = {}
for state in states:
    state_profiles[state] = [
        USHouseholdProfile.random(state=state, seed=i, strategy='realistic')
        for i in range(100)
    ]

print(f'  {"Metric":<35}', end='')
for state in states:
    print(f' {state:>10}', end='')
print()
print('-' * 68)

# Median income
print(f'  {"Median gross income/mo":<35}', end='')
for state in states:
    incomes = sorted(p.monthly_gross_income for p in state_profiles[state])
    print(f' ${incomes[50]:>8,.0f}', end='')
print()

# % renter (from census dist, not profiles)
print(f'  {"% renter (ACS)":<35}', end='')
for state in states:
    d = state_dists[state]
    val = f'{d.pct_renter:.1%}' if d else 'N/A'
    print(f' {val:>10}', end='')
print()

# Median gross rent (from census dist)
print(f'  {"Median gross rent/mo (ACS)":<35}', end='')
for state in states:
    d = state_dists[state]
    val = f'${d.median_gross_rent_monthly:,.0f}' if d else 'N/A'
    print(f' {val:>10}', end='')
print()

# SNAP receipt rate
print(f'  {"SNAP receipt rate (ACS)":<35}', end='')
for state in states:
    d = state_dists[state]
    val = f'{d.snap_receipt_rate:.1%}' if d else 'N/A'
    print(f' {val:>10}', end='')
print()

# % elderly or disabled
print(f'  {"% elderly/disabled (ACS)":<35}', end='')
for state in states:
    d = state_dists[state]
    val = f'{d.pct_elderly_or_disabled:.1%}' if d else 'N/A'
    print(f' {val:>10}', end='')
print()

# % citizen
print(f'  {"% citizen (ACS)":<35}', end='')
for state in states:
    d = state_dists[state]
    val = f'{d.pct_citizen:.1%}' if d else 'N/A'
    print(f' {val:>10}', end='')
print()

print()
print('States without bundled data use national fallback — their columns reflect')
print('national ACS weights, not true state demographics.')

In [ ]:
# Income distribution side-by-side (ASCII)
print('=== INCOME DISTRIBUTIONS BY STATE (100 realistic profiles each) ===')
print()
for state in states:
    incomes = sorted(p.monthly_gross_income for p in state_profiles[state])
    median = incomes[50]
    d = state_dists[state]
    source_note = 'state ACS' if d and d.state == state else 'national fallback'
    print(f'--- {state} ({source_note}) | median ${median:,.0f}/mo ---')
    buckets = [0] * 6
    edges = [0, 1000, 2500, 4000, 6000, 10000, 15001]
    labels = ['<$1k', '$1k-2.5k', '$2.5k-4k', '$4k-6k', '$6k-10k', '>$10k']
    for inc in incomes:
        for b in range(len(edges) - 1):
            if edges[b] <= inc < edges[b + 1]:
                buckets[b] += 1
                break
    for label, cnt in zip(labels, buckets):
        bar = '█' * cnt
        print(f'  {label:<12} {bar:<40} {cnt}')
    print()

## 5. Using the Realistic Strategy via the Pipeline

The `Pipeline.from_preset()` factory accepts a `profile_strategy` override. This lets you
swap in census-backed sampling without touching any generator code. The `realistic` strategy
is complementary to `edge_saturated` — use them for different evaluation goals:

- `edge_saturated` (default): stress-test reasoning at policy boundaries, maximize hard cases
- `realistic`: measure model coverage across the real applicant distribution
- `uniform`: simple baseline without any weighting

In [ ]:
from govsynth import Pipeline

pipeline = Pipeline.from_preset('snap.va', profile_strategy='realistic')
cases = pipeline.generate(n=50, seed=42)

print(f'Generated {len(cases)} cases with realistic strategy')
print()

outcomes = Counter(c.expected_outcome for c in cases)
diff     = Counter(c.difficulty.value  for c in cases)

print('Outcome distribution:')
for outcome, n in sorted(outcomes.items()):
    bar = '█' * n
    print(f'  {outcome:<12} {bar:<55} {n}')
print()
print('Difficulty distribution:')
for d_level, n in sorted(diff.items()):
    bar = '█' * n
    print(f'  {d_level:<12} {bar:<55} {n}')
print()
print('Compare to edge_saturated: realistic produces more clearly-eligible cases')
print('(households well under the income limit) and fewer hard boundary cases.')

In [ ]:
# Spot-check one case — confirm it has a realistic income level
case = cases[0]
print(f'Case ID:    {case.case_id}')
print(f'Outcome:    {case.expected_outcome}')
print(f'Scenario:   {case.scenario.summary}')
print()
print('Rationale trace:')
print(case.rationale_trace.to_plain_text())

## 6. Refreshing Census Data for a New State

Only Virginia ACS data is pre-bundled (`data/census/va.json`). To add a state, use the
`govsynth refresh-census-data` CLI command, which fetches ACS tables from the Census API
and writes a new JSON bundle:

```bash
# Fetch ACS 2022 5-year data for Texas and write data/census/tx.json
govsynth refresh-census-data --state TX

# Fetch for multiple states
govsynth refresh-census-data --state CA
govsynth refresh-census-data --state MD
```

**Requirements:**
- A free Census API key from [api.census.gov/data/key_signup.html](https://api.census.gov/data/key_signup.html)
- Set the key as `CENSUS_API_KEY` in your environment

Once `data/census/tx.json` exists, `CensusDataSource('TX').load()` will use it automatically
and the `realistic` strategy for TX profiles will draw from Texas ACS distributions.

**Virginia is always available without a network call** — it is committed to the repository
and loaded from the local JSON file. All other states fall back to national weights until
their state file is fetched.

The fetched files are deterministic for a given ACS vintage — commit them to version control
so that realistic profiles are reproducible across environments without requiring a Census API key.